# Oscillation: stock management and supply chains in Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/10-oscillation.ipynb)

This notebook is the Python companion to the [Oscillation: stock management and supply chains](https://waelrash1.github.io/loopbricks/#/learn/oscillation) module. You will build up an ordering rule one piece at a time, see how correcting a stock makes orders swing more than demand, find the cause of the cycle in the supply line, and measure how a longer delay makes the cycle slower and bigger. It uses three LoopBricks molecules: First-Order Stock Adjustment, Stock Management and Low-Visibility Pipeline Correction.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import matplotlib.pyplot as plt
print("loopbricks", lb.__version__, "with molecules:", ", ".join(lb.molecules()))

## 1. Replace what is used, then close the gap

Anyone managing a stock (a warehouse, a bank balance, a team) needs a rule for how fast to fill it. A good rule has two parts: **replace** what is being used up, and **correct** the gap between the desired and the actual level a little at a time. In **First-Order Stock Adjustment**, 10 units a month are used, the target is 100, and the level starts at 40.

In [ ]:
sa = lb.molecule("stockAdjust")
print(sa.equations(), "\n")
fig, ax = plt.subplots(figsize=(8, 4))
for label in ("Replace + correct", "No replacement ordering", "Slow adjustment"):
    run = sa.run(preset=label)
    run["Level"].plot(ax=ax, label=f"{label} {sa.preset(label)}")
    print(f"{label:<24}: level {run.loc[12.0, 'Level']:5.1f} in month 12, {run['Level'].iloc[-1]:5.1f} in month 60")
ax.axhline(100, ls=":", c="grey"); ax.set_ylabel("Level"); ax.legend(); ax.grid(alpha=0.3)

With both parts, the level climbs smoothly to 100. A slower adjustment gets there too, just later. But leave out replacement and the level gets stuck at 60. At that point the gap of 40, closed over 4 months, asks for 10 a month, which only just covers what is used, so the gap never closes. The shortfall is the usage times the adjustment time, 10 × 4 = 40: a **steady-state error**, built into the rule.

**Try it.** Without replacement, where does the level settle if the adjustment time is 2 months?

<details><summary>Show a worked answer</summary>

The shortfall is 10 × 2 = 20, so the level settles at 80, as the run below shows. Correcting faster shrinks the error but never removes it. (Try an adjustment time of 10: the rule of thumb predicts 0, but the run settles at about 9.1. Near empty, the stock cannot be used up faster than it can be drained in the minimum draining time of 1 month, so usage falls to Level / 1. The level settles where stocking (100 − Level) / 10 equals that usage, at 100 / 11. The shortfall formula holds only while the full usage of 10 can still be met.)
</details>

In [ ]:
for at in (2, 4, 10):
    print(f"adjustment time {at:>2}: level settles at {sa.run(replacement_fraction=0, adjustment_time=at)['Level'].iloc[-1]:.1f}")

## 2. Correcting a stock amplifies the flow

Now add a delay between ordering and receiving. **Stock Management** keeps orders in a supply line that takes `Acquisition delay` weeks to arrive, and its order rule has three parts: replace expected sales, correct the inventory gap, and correct the supply line gap.

To see the rule at work, we start the model in balance and raise demand from 10 to 12 a week in week 5. The supply line already starts full (demand × acquisition delay), but the inventory starts at 60, below its target. So we make a changed copy of the model: inventory starts at its desired value, and demand gets a `STEP`.

In [ ]:
sm = (lb.molecule("stockmgmt")
      .with_init("Inventory", "Desired_inventory")
      .with_equation("Demand", "10 + STEP(2, 5)"))     # demand rises by 2 in week 5

run = sm.run(T=80)
sm.plot(run, columns=["Sales", "Orders", "Acquisition"])
print(f"Demand rose by 2 a week; orders jumped by {run['Orders'].max() - 10:.1f}, to {run['Orders'].max():.1f} in week {run['Orders'].idxmax()}")
print(f"Inventory dipped to {run['Inventory'].min():.1f} in week {run['Inventory'].idxmin()}")

A 20% rise in demand makes orders jump by 50%, from 10 to 15. The order rule has to do three things at once: cover the new demand, rebuild the inventory that is now falling (it dips to 96.4), and fill a supply line that must hold more goods when more are flowing through it (12 × 6 = 72 instead of 60). Each link that manages a stock this way passes on a bigger swing than it receives. Chain several together and you have the **bullwhip effect**.

**Try it.** How big is the jump in orders if the acquisition delay is 3 weeks, and if it is 12?

<details><summary>Show a worked answer</summary>

Orders peak at 13.5 with a 3-week delay and 18 with a 12-week delay. The longer the pipeline, the more extra goods it must hold after demand rises (2 × the delay), and the rule tries to put them there in one adjustment time. Long lead times amplify demand changes.
</details>

In [ ]:
for acq in (3, 6, 12):
    o = sm.run(T=80, acquisition_delay=acq)["Orders"]
    print(f"acquisition delay {acq:>2}: orders peak at {o.max():.1f}")

## 3. The cause of the cycle

The rule above did not oscillate, because it counted the supply line. Real decision makers often do not: orders already placed are less visible than the shelf in front of them. **Low-Visibility Pipeline Correction** lets us set how much attention the pipeline gets (`Awareness of pipeline`, 0 to 1). The model starts with inventory at 60, below its target of 100.

In [ ]:
lv = lb.molecule("lowVisPipeline")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for aware in (1, 0.5, 0):
    run = lv.run(T=120, awareness_of_pipeline=aware)
    run["Inventory"].plot(ax=a1, label=f"awareness {aware}")
    run["Ordering"].plot(ax=a2, label=f"awareness {aware}")
    print(f"awareness {aware:<3}: inventory peaks at {run['Inventory'].max():5.1f}, orders fall as low as {run['Ordering'].min():4.1f}")
a1.set_title("Inventory"); a2.set_title("Ordering"); a1.axhline(100, ls=":", c="grey")
for a in (a1, a2): a.legend(); a.grid(alpha=0.3)

With full awareness, inventory rises smoothly to 100. With none, the manager keeps ordering to fill the gap on the shelf while plenty is already on its way; when it all arrives, inventory overshoots to 117 and orders are cut to 5.7, below the 10 a week that is shipped. That shortfall then pulls inventory down again, and the cycle repeats, slowly dying out. Half awareness gives a smaller swing. The oscillation comes from the structure, a balancing loop with a delay that the decision rule ignores, not from anything outside the system.

**Try it.** In the run with no awareness, how long is one cycle? Find the weeks where inventory reaches a peak.

<details><summary>Show a worked answer</summary>

The peaks are in weeks 16.4, 45.2, 74.1 and 103.0, so one cycle lasts about 29 weeks. That is several times the 6-week supplier delay: the period of a stock management cycle is set by the delay *and* by how fast the manager corrects.
</details>

In [ ]:
def peak_times(series):
    x, t = series.values, series.index
    return [float(t[i]) for i in range(1, len(x) - 1) if x[i - 1] < x[i] >= x[i + 1]]

inv = lv.run(T=120, awareness_of_pipeline=0)["Inventory"]
print("Inventory peaks in weeks", peak_times(inv))

## 4. Longer delays, slower and bigger swings

What does a slower supplier do to the cycle? Keep awareness at 0 and stretch the supplier delay.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for delay in (3, 6, 12):
    inv = lv.run(T=120, awareness_of_pipeline=0, supplier_delay=delay)["Inventory"]
    inv.plot(ax=ax, label=f"supplier delay {delay} weeks")
    pk = peak_times(inv)
    period = (pk[-1] - pk[0]) / (len(pk) - 1)
    print(f"delay {delay:>2}: inventory between {inv.min():5.1f} and {inv.max():5.1f}, one cycle about {period:.0f} weeks")
ax.axhline(100, ls=":", c="grey"); ax.set_ylabel("Inventory"); ax.legend(); ax.grid(alpha=0.3)

A longer delay makes the cycle both longer (from about 21 weeks to about 42) and larger: inventory overshoots to 106.5 with a 3-week supplier and to 130.5 with a 12-week one. More goods are in transit and unseen, so the manager over-orders for longer before the goods arrive. The two cures are to shorten the delay or to pay attention to the supply line; a manager can usually do the second far more cheaply than the first.

**Try it.** The *Boom and bust (slow supplier)* preset adds an impatient manager (inventory corrected in 2 weeks instead of 4) to a 12-week supplier who is ignored. How large is the swing, and what happens when the same manager tracks the pipeline (`awareness_of_pipeline=1`)?

<details><summary>Show a worked answer</summary>

Inventory swings from 58 up to 162, and orders range from 0 to 31 a week, when only 10 a week is shipped. The swings grow instead of dying out: the widest, from 58 to 162, both come after week 60. With full awareness the same impatient manager still starts by ordering 30 a week to close the gap of 40 quickly, and inventory overshoots once, to 104. Then the cycle dies: from week 60 on, inventory stays between 99.9 and 100.2. Correcting fast while ignoring the pipeline is the worst combination.
</details>

In [ ]:
for aware in (0, 1):
    run = lv.run(T=120, preset="Boom and bust (slow supplier)", awareness_of_pipeline=aware)
    late = run["Inventory"][run.index >= 60]
    print(f"awareness {aware}: inventory {run['Inventory'].min():5.1f} to {run['Inventory'].max():5.1f}, "
          f"orders {run['Ordering'].min():4.1f} to {run['Ordering'].max():4.1f}; "
          f"after week 60, inventory {late.min():5.1f} to {late.max():5.1f}")

## Where next

- Back in the app: the [Oscillation](https://waelrash1.github.io/loopbricks/#/learn/oscillation) module has the diagrams and more exercises, and the [beer game](https://waelrash1.github.io/loopbricks/#/learn/games/beer) lets you play the supply chain yourself.
- The delays behind these cycles are the subject of [Delays and perception](https://waelrash1.github.io/loopbricks/#/learn/delays).
- Inventory and workforce together drive business cycles: see [Cycles in business, labour and commodity markets](https://waelrash1.github.io/loopbricks/#/learn/cycles). For more on stock management, see Sterman 2000, ch. 17 and 18.